# Prophet — validation A100 et mesure R04

Ce notebook vérifie les sorties, états, gradients, le décodage et deux pas du trainer,
puis mesure les configurations R04 avec de vraies mises à jour d'optimiseur.
Les tokens aléatoires servent uniquement au profilage : aucune qualité de langage
ni supériorité architecturale n'est mesurée. Un échec du contrôle bloque le profilage.

Sélectionner un environnement GPU. Le notebook affiche la mémoire réellement disponible ;
il ne suppose pas que l'A100 possède 80 Go. Aucun montage Drive n'est nécessaire.
Les journaux et mesures restent dans `/content/Prophet_AGI/outputs` et doivent être
exportés avant de libérer l'environnement.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path

REVISION = "2340ea64529a4a8b23a2508524cfc4b156eaeb73"
repo = Path('/content/Prophet_AGI')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/speed25200-cyber/Prophet_AGI.git', str(repo)], check=True)
subprocess.run(['git', 'fetch', 'origin', 'claude/prophet-v03-memory-context'], cwd=repo, check=True)
subprocess.run(['git', 'checkout', REVISION], cwd=repo, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev,gpu]'], cwd=repo, check=True)

import torch
assert torch.cuda.is_available(), 'Sélectionner un environnement GPU dans Colab'
print('revision', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip())
print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 1024**3, 'GiB')
env = {**os.environ, 'TRITON_F32_DEFAULT': 'tf32x3', 'TRITON_CACHE_DIR': '/content/prophet-triton-tf32x3'}
(repo/'outputs').mkdir(exist_ok=True)

def run_logged(args, name):
    path = repo/'outputs'/name
    with path.open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', *args], cwd=repo, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f'{name}: exit {code}; inspect the saved log')
    return path


In [ ]:
# La compilation initiale des noyaux peut prendre plusieurs minutes.
gpu_gate_passed = False
run_logged(['-c', 'import torch; from prophet.modeling.layers import HAS_FLA; assert torch.cuda.is_available() and HAS_FLA, \"CUDA and fused FLA are required\"'], 'gpu-preflight.txt')
run_logged(['-m', 'pytest', 'tests/test_gpu.py', '-v', '--tb=short'], 'gpu-validation.txt')
gpu_gate_passed = True


In [ ]:
assert globals().get('gpu_gate_passed', False), 'Exécuter la validation GPU avec succès avant le profilage'
for batch_size in (1, 8):
    for variant in ('loop', 'plain'):
        stem = f'r04-{variant}-batch{batch_size}'
        run_logged(['scripts/gpu_check.py', '--config', f'configs/prophet_r04_{variant}.json',
                    '--batch-size', str(batch_size), '--seq-len', '2048',
                    '--steps', '3' if batch_size == 1 else '5', '--tokens', '100000000',
                    '--json-output', f'outputs/{stem}.json'], f'{stem}.txt')
        print(json.loads((repo/'outputs'/f'{stem}.json').read_text()))


In [ ]:
assert globals().get('gpu_gate_passed', False)
for variant in ('loop', 'plain'):
    for chunk in (None, 512):
        stem = f'loss-{variant}-{chunk}'
        args = ['scripts/gpu_check.py', '--config', f'configs/prophet_r04_{variant}.json',
                '--batch-size', '8', '--seq-len', '2048', '--steps', '5',
                '--tokens', '100000000', '--json-output', f'outputs/{stem}.json']
        if chunk is not None:
            args += ['--loss-chunk-tokens', str(chunk)]
        run_logged(args, f'{stem}.txt')
        print(json.loads((repo/'outputs'/f'{stem}.json').read_text()))
